In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import sawtooth,square
import time

In [2]:
DAC_SR = 9.8304e9 #in GHz
DAC_amplitude = 2**15 #16 bit representation in 2's complement format
N_points = 2**18
MAX_POINTS = int(2**14)
t = 1/DAC_SR*np.arange(N_points)
print(N_points)
print(MAX_POINTS)
print(t)

262144
16384
[0.00000000e+00 1.01725260e-10 2.03450521e-10 ... 2.66663615e-05
 2.66664632e-05 2.66665649e-05]


In [ ]:
# 100 ns square pulse, repeated every REPETITION_PERIOD
# The hardware loops the waveform forever: the read address counts 0..MAX_POINTS and wraps,
# so one loop lasts (MAX_POINTS + 1) fabric clocks of 16 samples each (1.6276 ns per clock).
PULSE_DURATION = 100e-9         # pulse width
REPETITION_PERIOD = 5e-6        # time between pulse starts; 5 us is exactly 3072 clocks, max about 26.7 us
SAMPLES_PER_CLK = 16

pulse_samples = int(round(DAC_SR * PULSE_DURATION))                         # 983 samples = 99.996 ns
period_clocks = int(round(REPETITION_PERIOD * DAC_SR / SAMPLES_PER_CLK))    # clocks per loop
assert period_clocks <= 2**14, "the period must fit in one slot (16384 clocks = 26.7 us)"
assert pulse_samples < period_clocks * SAMPLES_PER_CLK, "pulse must be shorter than the period"
N_points = period_clocks * SAMPLES_PER_CLK      # samples played per loop
MAX_POINTS = period_clocks - 1                  # the counter runs 0..MAX_POINTS, i.e. MAX_POINTS + 1 clocks
t = 1/DAC_SR*np.arange(N_points)
# 1 for the 100 ns pulse, 0 for the rest of the period
gate = (np.arange(N_points) < pulse_samples).astype(float)

print("pulse width   :", pulse_samples, "samples", f"({pulse_samples/DAC_SR*1e9:.3f} ns)")
print("loop period   :", N_points, "samples", f"({N_points/DAC_SR*1e6:.4f} us)")
print("MAX_POINTS    :", MAX_POINTS)

In [4]:
def dBm2Vpp(dbm):
    return 0.63245*10**(dbm/20)

print(dBm2Vpp(-6))
print(2*np.sqrt(100)*10**(-3/2))

0.3169758659075683
0.6324555320336758


In [ ]:
#Generating test signal: simple square pulses
PULSE_LEVEL = 0.4   # pulse height as a fraction of DAC full scale (1.0 = 2**15 codes)
freq = 1.02e9 #in Hz
num_of_zeros = 2**18-N_points
####### waveforms here
#DAC_B
mysignal_0z = PULSE_LEVEL*DAC_amplitude*gate
mysignal_0z = np.concatenate((mysignal_0z,np.zeros(num_of_zeros))) #Zero padding 

mysignal_0x = PULSE_LEVEL*DAC_amplitude*gate
mysignal_0x = np.concatenate((mysignal_0x,np.zeros(num_of_zeros))) #Zero padding 

#DAC_A
mysignal_2z = PULSE_LEVEL*DAC_amplitude*gate
mysignal_2z = np.concatenate((mysignal_2z,np.zeros(num_of_zeros))) #Zero padding 

mysignal_2x = PULSE_LEVEL*DAC_amplitude*gate
mysignal_2x = np.concatenate((mysignal_2x,np.zeros(num_of_zeros))) #Zero padding 

####################
###Rearranging samples in the format the hardware requires###


total_waveform_0 = np.append(mysignal_0z,mysignal_0x)

mask_0 = ([True]*8+[False]*8)*(len(total_waveform_0)//16)
mask_1 = ([False]*8+[True]*8)*(len(total_waveform_0)//16)

RAM_0 = total_waveform_0[mask_0]
RAM_1 = total_waveform_0[mask_1]
mysignal_0 = np.append(RAM_0,RAM_1)

total_waveform_2 = np.append(mysignal_2z,mysignal_2x)

mask_2 = ([True]*8+[False]*8)*(len(total_waveform_2)//16)
mask_3 = ([False]*8+[True]*8)*(len(total_waveform_2)//16)

RAM_2 = total_waveform_2[mask_2]
RAM_3 = total_waveform_2[mask_3]
mysignal_1 = np.append(RAM_2,RAM_3)

mysignal = np.append(mysignal_0,mysignal_1)
# round and clip to the signed 16-bit range (a PULSE_LEVEL above 1.0 would otherwise wrap around)
mysignal = np.clip(np.round(mysignal), -2**15, 2**15 - 1).astype(np.int16)

In [ ]:
# Preview (optional, not needed on the board): the start of each slot, and two loops of ch0 z
n_show = int(150e-9 * DAC_SR)
t_ns = np.arange(n_show) / DAC_SR * 1e9
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
for name, sig in [("ch0 z", mysignal_0z), ("ch0 x", mysignal_0x), ("ch2 z", mysignal_2z), ("ch2 x", mysignal_2x)]:
    ax1.plot(t_ns, sig[:n_show], label=name)
ax1.set_xlabel("time (ns)")
ax1.set_ylabel("DAC code")
ax1.legend(loc="upper right")
two_loops = np.tile(mysignal_0z[:N_points], 2)
ax2.plot(np.arange(len(two_loops)) / DAC_SR * 1e6, two_loops)
ax2.set_xlabel("time (us)")
ax2.set_title("two loops of ch0 z")
plt.tight_layout()
plt.show()

In [8]:
len(mysignal)/2**20

1.0

In [ ]:
RAM_1

### Load the overlay into the FPGA and program the oscillators on the board to produce the appropriate clock signals. 
The LMK and LMX files were taken from https://github.com/Xilinx/RFSoC-PYNQ/tree/master/boards/RFSoC4x2/packages/tics/tics/register_txts

In [97]:
from pynq import Overlay
import xrfclk
from pynq import PL
import pynq
from pynq.lib import AxiGPIO
PL.reset()
ol = Overlay("./awg_clock_syncd.bit") #Load the FPGA bit file 
xrfclk.set_ref_clks(lmk_freq = 245.76,lmx_freq = 491.52) #Programs the oscillators on the board to produce the appropriate clock signals

In [113]:
rst_n_ip = ol.ip_dict['rst_n']
rst_n =  AxiGPIO(rst_n_ip).channel1
print("Asserting extenal reset (low)...")
rst_n.write(0,1)
time.sleep(0.1)
print('Releasing external reset (high)...')
rst_n.write(1,1)

write_enable_ip = ol.ip_dict['write_enable']
write_enable = AxiGPIO(write_enable_ip).channel1
write_enable.write(1,0x1)

dma = ol.axi_dma_0
ndata = len(mysignal)
buf = pynq.allocate(shape = (ndata),dtype = np.uint16)
buf[:] = mysignal[:]
start = time.time()
dma.sendchannel.transfer(buf)
dma.sendchannel.wait()
stop = time.time()
print(f"Time spent wiriting : {stop - start}s")

max_instance = ol.ip_dict['MAX_POINTS']
max_port = AxiGPIO(max_instance).channel1
max_port.write(int(MAX_POINTS),0xffffffff)

axi_control_instance = ol.ip_dict['axi_control']
axi_control = AxiGPIO(axi_control_instance).channel1
axi_control.write(0,0x1)

enable_ch0_instance = ol.ip_dict['enable_ch0']
enable_ch0 = AxiGPIO(enable_ch0_instance).channel1

enable_ch2_instance = ol.ip_dict['enable_ch2']
enable_ch2 = AxiGPIO(enable_ch2_instance).channel1

write_enable.write(0,0x1)

Asserting extenal reset (low)...
Releasing external reset (high)...
Time spent wiriting : 0.0070688724517822266s


In [ ]:
# axi_control picks the slot that plays: 0 = z (waveforms A/C), 1 = x (waveforms B/D)
# enable_ch0 / enable_ch2 start the output, which then loops until disabled
axi_control.write(1,0x1)
enable_ch2.write(1,0x1)
enable_ch0.write(1,0x1)

In [15]:
write_enable.write(0,0x1)

In [17]:
from pynq import ps

print("CPU Clock Frequency:", ps.Clocks.cpu_mhz, "MHz")
print("FCLK0 Clock Frequency:", ps.Clocks.fclk0_mhz, "MHz")
print("FCLK1 Clock Frequency:", ps.Clocks.fclk1_mhz, "MHz")
print("FCLK2 Clock Frequency:", ps.Clocks.fclk2_mhz, "MHz")
print("FCLK3 Clock Frequency:", ps.Clocks.fclk3_mhz, "MHz")

CPU Clock Frequency: 1199.988 MHz
FCLK0 Clock Frequency: 214.283571 MHz
FCLK1 Clock Frequency: 99.999 MHz
FCLK2 Clock Frequency: 99.999 MHz
FCLK3 Clock Frequency: 99.999 MHz
